### Cell 1: Start Spark

In [1]:
from pathlib import Path
from pyspark.sql import SparkSession


def find_repo_root(start: Path) -> Path:
    """Walk up from start until a folder containing requirements.txt is found."""
    for folder in [start, *start.parents]:
        if (folder / "requirements.txt").is_file():
            return folder
    raise FileNotFoundError(f"No requirements.txt found in {start} or any parent folder")


# __file__ is not defined inside Jupyter, so fall back to the working directory
try:
    START = Path(__file__).resolve().parent
except NameError:
    START = Path.cwd().resolve()

ROOT = find_repo_root(START)
DATA = ROOT / "data"
MODELS = ROOT / "models"
print(f"ROOT = {ROOT}")

spark = (SparkSession.builder
         .appName("agnews-training")
         .config("spark.driver.memory", "4g")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
spark.version

ROOT = C:\Users\fakoy\CS4371-Group-Project-1


'3.5.9'

### Cell 2: Download AG News and save as Parquet

In [2]:
from datasets import load_dataset

ag = load_dataset("fancyzhx/ag_news")
ag["train"].to_parquet(str(DATA / "agnews_train.parquet"))
ag["test"].to_parquet(str(DATA / "agnews_test.parquet"))

C:\Users\fakoy\.conda\envs\hn-nlp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 83.96ba/s]


1879474

### Cell 3: Load into Spark and report class counts

In [3]:
from pyspark.sql import functions as F

train = spark.read.parquet(str(DATA / "agnews_train.parquet"))
test = spark.read.parquet(str(DATA / "agnews_test.parquet"))

label_names = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
name_col = F.create_map([F.lit(x) for kv in label_names.items() for x in kv])

for name, df in [("Train", train), ("Test", test)]:
    print(f"{name}: {df.count()} articles")
    (df.withColumn("class", name_col[F.col("label")])
       .groupBy("label", "class").count()
       .orderBy("label")
       .show())

Train: 120000 articles
+-----+--------+-----+
|label|   class|count|
+-----+--------+-----+
|    0|   World|30000|
|    1|  Sports|30000|
|    2|Business|30000|
|    3|Sci/Tech|30000|
+-----+--------+-----+

Test: 7600 articles
+-----+--------+-----+
|label|   class|count|
+-----+--------+-----+
|    0|   World| 1900|
|    1|  Sports| 1900|
|    2|Business| 1900|
|    3|Sci/Tech| 1900|
+-----+--------+-----+



### Cell 4: Build the Pipeline

In [4]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, CountVectorizer, IDF
from pyspark.ml.classification import LogisticRegression

tokenizer = RegexTokenizer(inputCol="text", outputCol="tokens",
                           pattern="\\W+", toLowercase=True, minTokenLength=2)
remover = StopWordsRemover(inputCol="tokens", outputCol="filtered")
vectorizer = CountVectorizer(inputCol="filtered", outputCol="tf",
                             vocabSize=50000, minDF=5)
idf = IDF(inputCol="tf", outputCol="features")
lr = LogisticRegression(featuresCol="features", labelCol="label",
                        maxIter=50, regParam=0.01)

pipeline = Pipeline(stages=[tokenizer, remover, vectorizer, idf, lr])

### Cell 5: Train, evaluate, and save

In [5]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit

grid = (ParamGridBuilder()
        .addGrid(lr.regParam, [0.001, 0.01, 0.1])
        .addGrid(vectorizer.vocabSize, [20000, 50000])
        .build())

tvs = TrainValidationSplit(estimator=pipeline,
                           estimatorParamMaps=grid,
                           evaluator=MulticlassClassificationEvaluator(
                               labelCol="label", metricName="accuracy"),
                           trainRatio=0.9, seed=42)

tvs_model = tvs.fit(train)
model = tvs_model.bestModel          # this is a PipelineModel you can save
print(tvs_model.validationMetrics)   # one score per grid combination

[0.875135450529299, 0.8742185546386597, 0.897474368592148, 0.8953071601233642, 0.910977744436109, 0.9118112861548721]


### Two-Stage Hyperparameter Tuning

Hyperparameters were tuned in two stages using `TrainValidationSplit`
(90/10 split of the training set, `seed=42`) with validation accuracy as the
metric. The test set was not used for any tuning decisions.

**Stage 1: initial grid.** A coarse search over regularization strength and
vocabulary size. The strongest regularization tested (0.1) performed best,
and it was at the edge of the grid, which suggested larger values might do better.

| regParam | vocabSize | Validation accuracy |
|----------|-----------|---------------------|
| 0.001 | 20000 | 0.8755 |
| 0.001 | 50000 | 0.8742 |
| 0.01  | 20000 | 0.8970 |
| 0.01  | 50000 | 0.8953 |
| 0.1   | 20000 | 0.9111 |
| 0.1   | 50000 | **0.9118** |

**Stage 2: refinement search.** Because the best regParam sat at the upper edge
of the Stage 1 grid, the next cell extends the search to larger values. Vocabulary
size is fixed at 20000, since it made little difference in Stage 1 (within 0.002
at each regParam).

| regParam | vocabSize | Validation accuracy |
|----------|-----------|---------------------|
| 0.1 | 20000 | 0.9106 |
| 0.2 | 20000 | 0.9121 |
| 0.5 | 20000 | **0.9145** |
| 1.0 | 20000 | 0.9104 |

The refinement did not confirm 0.1 as optimal: regParam=0.5 scored highest,
and its validation accuracy (0.9145) also beat the Stage 1 best (0.9118). The
final model therefore uses **regParam=0.5, vocabSize=20000**, replacing the
Stage 1 choice of regParam=0.1, vocabSize=50000. Accuracy falls off again at
1.0, so 0.5 sits inside the searched range rather than at its edge.

The gain is small (+0.0027). Across repeated runs, validation scores for the
same settings varied by up to about 0.001 because of nondeterminism in Spark's
optimizer, so the improvement is modest but larger than that noise. The final
model was evaluated on the test set only once, after selection.

In [6]:
refine_grid = (ParamGridBuilder()
               .addGrid(lr.regParam, [0.1, 0.2, 0.5, 1.0])
               .addGrid(vectorizer.vocabSize, [20000])
               .build())

refine_tvs = TrainValidationSplit(estimator=pipeline,
                                  estimatorParamMaps=refine_grid,
                                  evaluator=MulticlassClassificationEvaluator(
                                      labelCol="label", metricName="accuracy"),
                                  trainRatio=0.9, seed=42)

refine_model = refine_tvs.fit(train)

print("regParam  vocabSize  val_accuracy")
for params, score in zip(refine_grid, refine_model.validationMetrics):
    print(f"{params[lr.regParam]:<9} {params[vectorizer.vocabSize]:<10} {score:.4f}")

regParam  vocabSize  val_accuracy
0.1       20000      0.9103
0.2       20000      0.9121
0.5       20000      0.9141
1.0       20000      0.9106


In [7]:
# Pick the final model: whichever tuning stage has the best validation accuracy
stage1_best = max(tvs_model.validationMetrics)
stage2_best = max(refine_model.validationMetrics)

if stage2_best > stage1_best:
    model, chosen = refine_model.bestModel, "refinement (stage 2)"
else:
    model, chosen = tvs_model.bestModel, "initial grid (stage 1)"

best_lr = model.stages[-1]
best_cv = model.stages[2]
print(f"Stage 1 best val accuracy: {stage1_best:.4f}")
print(f"Stage 2 best val accuracy: {stage2_best:.4f}")
print(f"Final model from {chosen}: regParam={best_lr.getRegParam()}, "
      f"vocabSize={best_cv.getVocabSize()}")

Stage 1 best val accuracy: 0.9118
Stage 2 best val accuracy: 0.9141
Final model from refinement (stage 2): regParam=0.5, vocabSize=20000


In [8]:
# model was selected in the previous cell from the better tuning stage
test_preds = model.transform(test)

for metric in ["accuracy", "f1"]:
    ev = MulticlassClassificationEvaluator(labelCol="label",
                                           predictionCol="prediction",
                                           metricName=metric)
    print(f"Test {metric}: {ev.evaluate(test_preds):.4f}")

model.write().overwrite().save(str(MODELS / "agnews_lr_pipeline"))

Test accuracy: 0.9118
Test f1: 0.9116


In [9]:
(test_preds.groupBy("label").pivot("prediction").count()
           .orderBy("label").show())

+-----+----+----+----+----+
|label| 0.0| 1.0| 2.0| 3.0|
+-----+----+----+----+----+
|    0|1694|  58|  98|  50|
|    1|  12|1870|   9|   9|
|    2|  59|  17|1662| 162|
|    3|  55|  13| 128|1704|
+-----+----+----+----+----+



### Confusion Matrix (Test Set)

Final model: regParam=0.5, vocabSize=20000 (selected on validation accuracy).

Overall test accuracy: 91.2% (6,930 / 7,600)

Sports is the strongest class (98.4% recall), with vocabulary that rarely
overlaps the others. The main confusion is between Business and Sci/Tech:
160 Business articles were predicted as Sci/Tech and 128 went the other way,
accounting for 288 of 670 errors (43%). This boundary is relevant for Hacker News
predictions, where most content sits at the intersection of technology and
business news.

In [10]:
order = ["World", "Sports", "Business", "Sci/Tech"]

named = (test_preds
         .withColumn("true_class", name_col[F.col("label")])
         .withColumn("pred_class", name_col[F.col("prediction").cast("int")]))

(named.groupBy("true_class")
      .pivot("pred_class", order)
      .count()
      .withColumn("_order", 
          F.when(F.col("true_class") == "World", 0)
           .when(F.col("true_class") == "Sports", 1)
           .when(F.col("true_class") == "Business", 2)
           .otherwise(3))
      .orderBy("_order")
      .drop("_order")
      .show())

+----------+-----+------+--------+--------+
|true_class|World|Sports|Business|Sci/Tech|
+----------+-----+------+--------+--------+
|     World| 1694|    58|      98|      50|
|    Sports|   12|  1870|       9|       9|
|  Business|   59|    17|    1662|     162|
|  Sci/Tech|   55|    13|     128|    1704|
+----------+-----+------+--------+--------+



In [11]:
import json

ag_metrics = {}
for metric in ["accuracy", "f1"]:
    ev = MulticlassClassificationEvaluator(labelCol="label",
                                           predictionCol="prediction",
                                           metricName=metric)
    ag_metrics[metric] = round(float(ev.evaluate(test_preds)), 4)
    print(f"Test {metric}: {ag_metrics[metric]:.4f}")

with open(str(DATA / "agnews_test_metrics.json"), "w") as f:
    json.dump(ag_metrics, f, indent=2)
print("Saved AG News metrics to data/agnews_test_metrics.json")

model.write().overwrite().save(str(MODELS / "agnews_lr_pipeline"))

Test accuracy: 0.9118
Test f1: 0.9116
Saved AG News metrics to data/agnews_test_metrics.json
